# Synthetic Dataset Generator
Generates CSV training datasets using OpenAI. Supports three dataset types with a Gradio UI.

In [ ]:
import os
import tempfile
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
client = OpenAI()
MODEL = 'gpt-4o-mini'

In [ ]:
COLUMNS = {
    "Instruction-Response Pairs": "instruction,response,domain,complexity",
    "Multi-Turn Chat Dialogue":   "conversation_id,turn_number,role,content",
    "Text Classification":        "text,label,style",
}

def build_prompt(dataset_type, num_samples, **kwargs):
    columns = COLUMNS[dataset_type]
    system = (
        f"You are a synthetic dataset generator. "
        f"Output ONLY valid CSV with header: {columns}. No extra text."
    )
    if dataset_type == "Instruction-Response Pairs":
        user = (f"Generate {num_samples} instruction-response pairs "
                f"for domain '{kwargs['domain']}', "
                f"instruction type '{kwargs['instruction_type']}', "
                f"complexity '{kwargs['complexity']}'.")
    elif dataset_type == "Multi-Turn Chat Dialogue":
        user = (f"Generate {num_samples} conversations "
                f"for scenario '{kwargs['scenario']}', "
                f"{kwargs['turns']} turns each, "
                f"goal: '{kwargs['goal']}'.")
    else:
        user = (f"Generate {num_samples} text classification samples "
                f"for text type '{kwargs['text_type']}', "
                f"labels: {kwargs['labels']}.")
    return system, user

def generate(dataset_type, num_samples, **kwargs):
    system, user = build_prompt(dataset_type, num_samples, **kwargs)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        stream=True,
    )
    result = ""
    for chunk in response:
        delta = chunk.choices[0].delta.content or ""
        result += delta
        yield result, gr.update(visible=False)

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".csv", mode="w", encoding="utf-8")
    tmp.write(result)
    tmp.close()
    yield result, gr.update(visible=True, value=tmp.name)

def generate_ir(nsamples, domain, instruction_type, complexity):
    yield from generate("Instruction-Response Pairs", int(nsamples),
                        domain=domain, instruction_type=instruction_type, complexity=complexity)

def generate_mtd(nsamples, scenario, turns, goal):
    yield from generate("Multi-Turn Chat Dialogue", int(nsamples),
                        scenario=scenario, turns=int(turns), goal=goal)

def generate_tc(nsamples, text_type, labels):
    yield from generate("Text Classification", int(nsamples),
                        text_type=text_type, labels=labels)

DATASET_TYPES = list(COLUMNS.keys())

def toggle(dataset_type):
    return (
        gr.update(visible=dataset_type == "Instruction-Response Pairs"),
        gr.update(visible=dataset_type == "Multi-Turn Chat Dialogue"),
        gr.update(visible=dataset_type == "Text Classification"),
    )

with gr.Blocks(title="Synthetic Dataset Generator") as demo:
    gr.Markdown("## Synthetic Dataset Generator")

    with gr.Row():
        dtype    = gr.Dropdown(DATASET_TYPES, label="Dataset Type")
        nsamples = gr.Slider(5, 50, value=10, step=5, label="Number of Samples")

    with gr.Group(visible=False) as ir_group:
        gr.Markdown("### Instruction-Response Pairs")
        with gr.Row():
            ir_domain = gr.Textbox(label="Domain", placeholder="e.g. Cooking")
            ir_type   = gr.Textbox(label="Instruction Type", placeholder="e.g. Summarization")
            ir_comp   = gr.Textbox(label="Complexity", placeholder="e.g. beginner to advanced")
        ir_btn = gr.Button("Generate", variant="primary")

    with gr.Group(visible=False) as mtd_group:
        gr.Markdown("### Multi-Turn Chat Dialogue")
        with gr.Row():
            mtd_scenario = gr.Textbox(label="Scenario", placeholder="e.g. customer support")
            mtd_turns    = gr.Slider(2, 10, value=3, step=1, label="Turns per Conversation")
            mtd_goal     = gr.Textbox(label="Conversation Goal", placeholder="e.g. resolve a complaint")
        mtd_btn = gr.Button("Generate", variant="primary")

    with gr.Group(visible=False) as tc_group:
        gr.Markdown("### Text Classification")
        with gr.Row():
            tc_type   = gr.Textbox(label="Text Type", placeholder="e.g. News Headlines")
            tc_labels = gr.Textbox(label="Labels (comma-separated)", placeholder="e.g. Sports, Politics, Tech")
        tc_btn = gr.Button("Generate", variant="primary")

    output   = gr.Textbox(label="Generated CSV", lines=15)
    download = gr.File(label="Download CSV", visible=False)

    dtype.change(toggle, inputs=dtype, outputs=[ir_group, mtd_group, tc_group])
    ir_btn.click(fn=generate_ir, inputs=[nsamples, ir_domain, ir_type, ir_comp], outputs=[output, download])
    mtd_btn.click(fn=generate_mtd, inputs=[nsamples, mtd_scenario, mtd_turns, mtd_goal], outputs=[output, download])
    tc_btn.click(fn=generate_tc, inputs=[nsamples, tc_type, tc_labels], outputs=[output, download])

demo.launch()